## 1. Environment Setup & Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

# Add project root to sys.path to import modular pipeline
sys.path.append(os.path.abspath("../"))
from src.feature_engineering import load_data, run_feature_engineering_pipeline, prepare_datasets

from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, 
                             median_absolute_error, mean_absolute_percentage_error)

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
import os
import pandas as pd
from src.feature_engineering import load_data, run_feature_engineering_pipeline, prepare_datasets

# Define paths
data_path = "../data/laptopData.csv"
spec_path = "../data/specData.csv"

# Calculate brand ratings from specData.csv (for improved accuracy)
brand_ratings = None
if os.path.exists(spec_path):
    print("Calculating brand ratings from spec data")
    spec_df = pd.read_csv(spec_path)
    spec_df['spec_rating'] = pd.to_numeric(spec_df['spec_rating'], errors='coerce')
    spec_df['brand_match'] = spec_df['brand'].astype(str).str.lower().str.strip()
    brand_ratings = spec_df.groupby('brand_match')['spec_rating'].mean().to_dict()

# Run the robust feature engineering pipeline with raw data
df_raw = load_data(data_path)
df_features = run_feature_engineering_pipeline(df_raw, brand_ratings=brand_ratings)
df_features = df_features.dropna()

# Split and Scale
X_train, X_test, y_train, y_test, scaler = prepare_datasets(df_features)

print(f"Data loaded from: {data_path}")
print(f"Total features after encoding: {X_train.shape[1]}")
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

## 2. Model Training & Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf'),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "CatBoost": CatBoostRegressor(n_estimators=100, verbose=0, random_state=42),
    "Stacked Generalizer": StackingRegressor(
        estimators=[
            ('xgb', XGBRegressor(n_estimators=100, random_state=42)),
            ('cat', CatBoostRegressor(n_estimators=100, verbose=0, random_state=42)),
            ('et', ExtraTreesRegressor(n_estimators=100, random_state=42)),
            ('svr', SVR(kernel='rbf', C=10))
        ],
        final_estimator=RidgeCV(),
        cv=5,
        n_jobs=-1
    )
}

def evaluate_models(models, X_train, X_test, y_train, y_test):
    results = []
    for name, model in models.items():
        start_time = time()
        model.fit(X_train, y_train)
        train_time = time() - start_time
        y_pred = model.predict(X_test)
        
        # Metrics on Log Price
        mse_log = mean_squared_error(y_test, y_pred)
        mae_log = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Metrics on Original Price (Exp)
        y_test_exp = np.exp(y_test)
        y_pred_exp = np.exp(y_pred)
        mae_price = mean_absolute_error(y_test_exp, y_pred_exp)
        
        results.append({
            "Model": name,
            "R2 Score": r2,
            "MAE (Log)": mae_log,
            "MAE (Price)": mae_price,
            "Time (s)": train_time
        })
        
    return pd.DataFrame(results)

results_df = evaluate_models(models, X_train, X_test, y_train, y_test)
results_df = results_df.sort_values(by="R2 Score", ascending=False)
results_df

To evaluate model performance, multiple metrics were used on both the log-transformed target (Log_Price) and the original price scale, ensuring both statistical accuracy and real-world interpretability. Tree Ensembles (Gradient Boosting, XGBoost, and Random Forest) are top performers because they are best at capturing complex interactions between your features. EDA showed significant multicollinearity and interaction effects between features, thus tree ensembles performed better. Additionally, tree-based models don't require features to be normally distributed, which is helpful since most people buy mid-weight laptops. SVR performed well because of its high-dimensional geometry. Kernel trick allows for SVR to project data into higher-dimensional space where non-linear patterns become easier to separate. Additionally, unlike linear regression, SVR is less sensitive to extreme outliers because it focuses on points close to the decision boundary, making it robust against luxuriously priced laptops that might skew the linear models. Linear models assume that each feature adds a fixed amount to price independently, so it fails to capture the "Brand Tax" or "Gaming Premium" features that have multiplicative impact on Price.

Why XGBoost outperforms Random Forest:
Random Forest uses bagging, so it is good for reducing variance but each tree is unaware of what the others missed. XGBoost uses boosting, so each tree focuses on residuals of the previous ones. This makes XGBoost better at predicting niche laptop categories. Additionally, XGBoost includes L1 and l2 penalties in its loss function. This is critical for this dataset, because we have over 40 one-hot-encoded features. Regularization prevents the model from assigning too much important to rare categories, ensuring it generalizes better to new data. The many one-hot-encoded features also mean that the data is highly sparse due to the dummy variables. XGBoost learns a default direction for missing values or zeroes, which is much more efficient than Random Forest's approach of trying every possible split.

In [ ]:
# Avg Error seemed way too high (~9K INR), so comparing to baseline model that predicts average
# price for every laptop
baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(np.exp(y_test), np.exp(baseline_pred))

print(baseline_mae)


It can be seen that MAE of baseline is 29100, compared to ~9000 of the tree-based ensembles.

## 3. Best Model Deep Dive

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Log Price")
plt.ylabel("Predicted Log Price")
plt.title(f"Predictions vs Actuals: {best_model_name}")
plt.show()

print(f"R² Score: {r2:.4f}")

## 4. Feature Importance

In [ ]:
if hasattr(best_xgb, 'feature_importances_'):
    feat_imp = pd.Series(best_xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='mako')
    plt.title('Top 15 Feature Importances (via Tuned XGBoost)')
    plt.show()

## 5. Hyperparameter Tuning for Top 4 Models
Tuning top-performing models (XGBoost, Random Forest, SVR, Gradient Boosting) using RandomizedSearchCV to find the best configuration for each. RandomizedSearchCV used over GridSearchCV as the gain from an exhaustive Grid Search is usually negligible compared to the time saved by a random search. 

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR

# Robust Hyperparameter Grids
rf_grid = {
    'n_estimators': [100, 300, 500], 
    'max_depth': [10, 20, None], 
    'max_features': ['sqrt', 'log2', None]
}
gb_grid = {
    'n_estimators': [100, 300, 500], 
    'learning_rate': [0.01, 0.05, 0.1], 
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.8, 0.9, 1.0]
}
xgb_grid = {
    'n_estimators': [100, 300, 500, 1000], 
    'learning_rate': [0.01, 0.05, 0.1], 
    'max_depth': [4, 6, 8, 10], 
    'subsample': [0.7, 0.8, 1.0]
}
cat_grid = {
    'iterations': [100, 500, 1000], 
    'learning_rate': [0.01, 0.05, 0.1], 
    'depth': [4, 6, 8, 10], 
    'l2_leaf_reg': [1, 3, 5, 7]
}
et_grid = {
    'n_estimators': [100, 300, 500], 
    'max_depth': [10, 20, None], 
    'min_samples_split': [2, 5, 10]
}
svr_grid = {
    'C': [0.1, 1, 10, 100, 500], 
    'epsilon': [0.01, 0.1, 0.2], 
    'gamma': ['scale', 'auto', 0.1, 0.01]
}

In [ ]:
def tune_model(estimator, param_dist, name):
    print(f"Tuning {name}...")
    # Using n_iter=20 for more thorough search to beat baseline results
    search = RandomizedSearchCV(
        estimator=estimator, 
        param_distributions=param_dist, 
        n_iter=20, 
        cv=5, 
        scoring='r2', 
        n_jobs=-1, 
        random_state=42
    )
    search.fit(X_train, y_train)
    # Record validation score
    print(f"Best {name} CV R2: {search.best_score_:.4f}")
    return search.best_estimator_

best_rf = tune_model(RandomForestRegressor(random_state=42), rf_grid, "Random Forest")
best_gb = tune_model(GradientBoostingRegressor(random_state=42), gb_grid, "Gradient Boosting")
best_xgb = tune_model(XGBRegressor(random_state=42), xgb_grid, "XGBoost")
best_cat = tune_model(CatBoostRegressor(verbose=0, random_state=42), cat_grid, "CatBoost")
best_et = tune_model(ExtraTreesRegressor(random_state=42), et_grid, "Extra Trees")
best_svr = tune_model(SVR(kernel='rbf'), svr_grid, "SVR")

In [ ]:
## Final Tuned Stacked Generalizer
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

print("Building Final Tuned Stacked Generalizer...")
final_tuned_stack = StackingRegressor(
    estimators=[
        ('xgb', best_xgb),
        ('cat', best_cat),
        ('et', best_et),
        ('svr', best_svr)
    ],
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1
)

final_tuned_stack.fit(X_train, y_train)

It can be seen that Gradient Boosting outperformed XGBoost after hyperparameter tuning. XGBoost is designed to already have default parameters quite optimized, which is why it was the best model prior to tuning. Gradient Boosting is very sensitive to learning rate and subsampling, so in the baseline it was likely taking steps that were too large or looking at too much data at once. During tuning, it found the optimal learning rate and depth that allowed it to learn the nuances of features more precisely than XGBoost. Additionally, XGBoost is built for big datasets as its complex internal regularization is designed to prevent overfitting. However, because this dataset is small, the extra complexity might have been a hindrance. It's possible that the 15 iterations found a near-perfect configuration for Gradient Boosting, whereas XGBoost has so many more "moving parts" (gamma, colsample_bytree, alpha, etc.) that 15 iterations only scratched the surface of its potential.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics import r2_score, mean_absolute_error

## 6. Final Performance Report
def evaluate_final(model, name):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    # Adjusted R2 Calculation
    n = X_test.shape[0]
    p = X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    mae_price = mean_absolute_error(np.exp(y_test), np.exp(y_pred))
    return {"Model": name, "R2": r2, "Adj R2": adj_r2, "MAE (Price)": mae_price}

final_results = [
    evaluate_final(best_rf, "Tuned Random Forest"),
    evaluate_final(best_gb, "Tuned Gradient Boosting"),
    evaluate_final(best_xgb, "Tuned XGBoost"),
    evaluate_final(best_cat, "Tuned CatBoost"),
    evaluate_final(best_et, "Tuned Extra Trees"),
    evaluate_final(best_svr, "Tuned SVR"),
    evaluate_final(final_tuned_stack, "Tuned Stacked Generalizer (Ensemble)")
]

final_df = pd.DataFrame(final_results).sort_values(by="R2", ascending=False)
final_df.to_csv("../results/final_tuned_performance.csv", index=False)

# Identify and save the winner
best_overall_name = final_df.iloc[0]["Model"]
print(f" WINNER: {best_overall_name}")

best_overall_model = None
if best_overall_name == "Tuned Stacked Generalizer (Ensemble)":
    best_overall_model = final_tuned_stack
elif "Random Forest" in best_overall_name: best_overall_model = best_rf
elif "Gradient Boosting" in best_overall_name: best_overall_model = best_gb
elif "XGBoost" in best_overall_name: best_overall_model = best_xgb
elif "CatBoost" in best_overall_name: best_overall_model = best_cat
elif "Extra Trees" in best_overall_name: best_overall_model = best_et
elif "SVR" in best_overall_name: best_overall_model = best_svr

if best_overall_model:
    os.makedirs("../models", exist_ok=True)
    joblib.dump(best_overall_model, "../models/best_laptop_price_model_final.joblib")

final_df

In [ ]:
## 7. Visualizing Best Model Fit
y_pred_final = final_tuned_stack.predict(X_test)
r2_final = r2_score(y_test, y_pred_final)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_final, alpha=0.5, color='royalblue', label=f'Stacked Ensemble R² = {r2_final:.4f}')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel("Actual Price (Log Scale)")
plt.ylabel("Predicted Price (Log Scale)")
plt.title("Final Tuned Ensemble: Performance Visualization")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()